# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [41]:
%load_ext dotenv
%dotenv ../05_src/.secrets

import os
from dotenv import load_dotenv

load_dotenv("../05_src/.secrets", override=True)

print(repr(os.getenv("OPENAI_API_KEY")))

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv
'any_value'


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [42]:
from langchain_community.document_loaders import PyPDFLoader
file_path = "./documents/ai_report_2025.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

print(len(docs))

document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

print(len(document_text))
print(document_text[0:50])

26
53851
pg. 1 
 
 
The GenAI Divide  
STATE OF AI IN 
BUSI


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


Connect to openAI's API

In [43]:
# connect to openAI's API
from openai import OpenAI
import os
client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
                api_key='any value',
                default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})

Construct structured output using Pydantic model

In [44]:
from pydantic import BaseModel

class ArticleSummary(BaseModel):
    author: str
    title: str
    relevance: str
    summary: str
    tone: str
    InputTokens: int
    OutputTokens: int

Add detailed instructions to the prompt

In [45]:
# construct the prompt
prompt = f"""
    Given the following context from an article, do the following:
    
    1. Identify the book's author and title.
    2. Determine the relevance of the article, give a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    and tone of the article.
    3. Summarize the article, give a concise and succinct summary no longer than 1000 tokens.
    4. Determine the tone of the summary.
        
    The article is the following: 
    <article>
    {document_text}
    </article>
"""

Add a developer prompt

In [46]:
system_prompt = "You are summarizing only in a formal academic writing tone"

Check the response

In [47]:
response = client.responses.parse(
    model='gpt-4o',
    instructions=system_prompt,
    input=prompt,
    text_format=ArticleSummary,
)

In [ ]:
result = response.output_parsed
result.InputTokens = response.usage.input_tokens
result.OutputTokens = response.usage.output_tokens

Display the output

In [50]:
result
result.model_dump()

{'author': 'Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari',
 'title': 'The GenAI Divide: State of AI in Business 2025 - Project NANDA',
 'relevance': 'This article is pivotal for AI professionals as it highlights the challenges in AI implementation and the GenAI Divide, emphasizing the importance of adaptive systems and strategic partnerships for successful AI integration, thus guiding professionals in optimizing AI deployment strategies.',
 'summary': 'The report, "The GenAI Divide: State of AI in Business 2025" by Project NANDA, presents findings on AI adoption in enterprises, revealing that despite substantial investment, 95% of organizations do not achieve a measurable return on AI initiatives—a phenomenon termed the GenAI Divide. This divide is marked by a high adoption rate but low transformation, primarily due to non-learning AI systems that fail to integrate effectively into workflows. While tools like ChatGPT are popular for enhancing productivity, enterprise

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

## Summarization Metric

Set up model, summarization metric, and test case using the DeepEval library

In [63]:
from deepeval.models import GPTModel
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.metrics import GEval, SummarizationMetric

# construct model
model = GPTModel(
    model="gpt-4o-mini",
    temperature=0,
    _openai_api_key='any value',
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1'
)

# add assessment questions
# need to read the document to come up with bespoke questions
questions=[
        "Does the summary reflect the main idea of the article?",
        "Does the summary mention GenAI Divide?",
        "Does the summary mention GenAI's impact on business?",
        "Does the summary address the concept of shadow AI?",
        "Does the summary mention how to bridge the GenAI Divide?"
    ]

# construct the metric
summarization_metric = SummarizationMetric(
    assessment_questions=questions,
    model=model,
    include_reason=True,
    threshold=0.5
)

# add test case
test_case = LLMTestCase(
    input=document_text,
    actual_output=result.summary
)

# get metrics
summarization_metric.measure(test_case)

Output()

0.26666666666666666

## G-Eval Metrics

Implement additional evalution metrics for Coherence, Tonality and Safety using G-Eval

In [65]:
coherence_metric = GEval(
    name="Coherence",
    evaluation_steps=[
        "Check if ideas flow logically from one to the next",
        "Check if the summary contradicts itself",
        "Check if sentences connect clearly to each other",
        "Penalize abrupt topic changes",
        "Check if the summary has a clear structure"
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model
)

tonality_metric = GEval(
    name="Tonality",
    evaluation_steps=[
        "Check if the tone is consistently formal and academic",
        "Penalize casual or conversational language",
        "Check if the vocabulary is scholarly",
        "Check if the tone is consistent throughout",
        "Check if hedging language is used appropriately"
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model
)

safety_metric = GEval(
    name="Safety",
    evaluation_steps=[
        "Check if the summary contains harmful claims",
        "Check if the summary misrepresents the article",
        "Check if the summary introduces unsupported facts",
        "Penalize content that could spread misinformation",
        "Check if predictions are clearly supported by the article"
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model
)

In [66]:
# show metrics
print(coherence_metric.measure(test_case))
print(tonality_metric.measure(test_case))
print(safety_metric.measure(test_case))

Output()

Output()

0.794177017978822


Output()

0.8917758028688165


0.8717782170026505


## Output the metrics

In [69]:
class EvaluationResult(BaseModel):
    SummarizationScore: float
    SummarizationReason: str
    CoherenceScore: float
    CoherenceReason: str
    TonalityScore: float
    TonalityReason: str
    SafetyScore: float
    SafetyReason: str

evaluation_scores = EvaluationResult(
    SummarizationScore=summarization_metric.score,
    SummarizationReason=summarization_metric.reason,
    CoherenceScore=coherence_metric.score,
    CoherenceReason=coherence_metric.reason,
    TonalityScore=tonality_metric.score,
    TonalityReason=tonality_metric.reason,
    SafetyScore=safety_metric.score,
    SafetyReason=safety_metric.reason,
)

evaluation_scores.model_dump()

{'SummarizationScore': 0.26666666666666666,
 'SummarizationReason': 'The score is 0.27 because the summary contains significant contradictions to the original text, particularly regarding authorship, and includes numerous pieces of extra information that were not present in the original text, leading to a lack of accuracy and completeness.',
 'CoherenceScore': 0.794177017978822,
 'CoherenceReason': 'The response presents a logical flow of ideas, starting with the introduction of the GenAI Divide and moving through the challenges and solutions related to AI adoption. However, there are minor abrupt transitions, particularly when shifting from discussing industry-specific disruptions to the barriers of scaling AI, which could be smoother. Overall, the structure is clear, and the sentences connect well, but slight improvements in topic transitions would enhance coherence.',
 'TonalityScore': 0.8917758028688165,
 'TonalityReason': 'The response maintains a formal and academic tone througho

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

Imporve the original prompt based on the evaluation. Add instruction to improve the summarization score, i.e., "Make sure the summary is accurate and there is no contradiactions to the original text".

In [ ]:
# construct a better prompt
prompt = f"""
    Given the following context from an article, do the following:
    
    1. Identify the book's author and title.
    2. Determine the relevance of the article, give a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    and tone of the article.
    3. Summarize the article, give a concise and succinct summary no longer than 1000 tokens. Make sure the summary is accurate and there is no contradiactions to the original text.
    4. Determine the tone of the summary.
        
    The article is the following: 
    <article>
    {document_text}
    </article>
"""

Show the new response

In [70]:
response = client.responses.parse(
    model='gpt-4o',
    instructions=system_prompt,
    input=prompt,
    text_format=ArticleSummary,
)

result_v2 = response.output_parsed
result_v2.InputTokens = response.usage.input_tokens
result_v2.OutputTokens = response.usage.output_tokens
result_v2.model_dump()


{'author': 'Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari',
 'title': 'The GenAI Divide: State of AI in Business 2025',
 'relevance': 'The article presents a comprehensive analysis of the barriers to successful AI implementation, offering insights into the GenAI Divide that separates the minority of successful endeavors from widespread failures. It is essential for AI professionals to understand these dynamics to enhance AI strategies, improve decision-making regarding investment, and develop systems that are truly transformative rather than merely operational.',
 'summary': "The report, authored by Aditya Challapally and his colleagues, explores the stark divide in AI implementation success across enterprises. Despite significant investment in Generative AI, only a small fraction of pilots yield substantial returns, attributable to the fundamental gap in learning and adaptation capabilities of AI systems. Many organizations find themselves 'on the wrong side' of the 

Build new test case for the new response

In [71]:
# add test case
test_case_v2 = LLMTestCase(
    input=document_text,
    actual_output=result_v2.summary
)

# rerun all metrics
summarization_metric.measure(test_case_v2)
coherence_metric.measure(test_case_v2)
tonality_metric.measure(test_case_v2)
safety_metric.measure(test_case_v2)

evaluation_scores_v2 = EvaluationResult(
    SummarizationScore=summarization_metric.score,
    SummarizationReason=summarization_metric.reason,
    CoherenceScore=coherence_metric.score,
    CoherenceReason=coherence_metric.reason,
    TonalityScore=tonality_metric.score,
    TonalityReason=tonality_metric.reason,
    SafetyScore=safety_metric.score,
    SafetyReason=safety_metric.reason,
)

evaluation_scores_v2.model_dump()

Output()

Output()

Output()

Output()

{'SummarizationScore': 0.46153846153846156,
 'SummarizationReason': 'The score is 0.46 because the summary contradicts the original text by claiming that more industries are experiencing disruption from GenAI, while the original states only two are affected. Additionally, the summary includes several points not found in the original text, such as authorship of the report and details on industry resistance, which detracts from its accuracy and relevance.',
 'CoherenceScore': 0.8020185483236671,
 'CoherenceReason': 'The response presents a logical flow of ideas, discussing the divide in AI implementation and the factors contributing to it. Sentences connect well, and the structure is clear, moving from the problem to potential solutions. However, there is a slight abruptness in transitioning between the discussion of industry-specific resistance and the emergence of a shadow AI economy, which could be better integrated for smoother reading.',
 'TonalityScore': 0.8562176500885798,
 'Tonal

**The Summarization score improved a lot from 0.26 to 0.46, meaning the new prompt with addictional instruction is improving the text summary.**

**I don't think these control are enough. I notice that when running evaluation metrics multiple times, the scores are different, meaning the test itself is not deterministic. We need to think of a way to make sure the tests are working as intended and generating reliable scores.**


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
